## setup

In [ ]:
# import stuff
import pandas as pd
import numpy as np

# roberta sentiment analysis
# %pip install -U transformers
# %pip install torch
from transformers import pipeline

## eda

In [ ]:
# get data
df = pd.read_csv('data/reputation_items_all_sources.csv')

print('Shape:', df.shape)
df.head()

In [ ]:
df.describe()

In [ ]:
df['rating'].value_counts()

In [ ]:
df.isna().sum()

assuming we get reddit data, those reviews will populate the title and engagement fields

In [ ]:
df.dtypes

In [ ]:
df[['source', 'clinic', 'collection_method']].value_counts()

In [ ]:
# setup pipeline for sentiment analysis using pre-trained model
classifier = pipeline(
    "sentiment-analysis",
    model = "cardiffnlp/twitter-roberta-base-sentiment-latest",
    framework = "pt",
    device = -1
    )

In [ ]:
df['text'] = df['text'].astype(str)

df['text'].head(3)

In [ ]:
results = classifier(
    df['text'].tolist(),
    truncation = True,
    max_length = 512,
    top_k = None,
    batch_size = 32
)

results[0]

In [ ]:
# get pos/neg/neu labels + their probabilities for each review
probs = [{r["label"].lower(): r["score"] for r in res} for res in results]

# get pos/neg/neu probability breakdowns
df["positive_prob"] = [p["positive"] for p in probs]
df["neutral_prob"] = [p["neutral"] for p in probs]
df["negative_prob"] = [p["negative"] for p in probs]

# get overall confidence & predicted sentiment
prob_cols = ["positive_prob", "neutral_prob", "negative_prob"]
df["model_confidence"] = df[prob_cols].max(axis = 1)
df["predicted_sentiment"] = df[prob_cols].idxmax(axis = 1).str.replace("_prob", "")

# get overall sentiment
df["overall_sentiment"] = df["positive_prob"] - df["negative_prob"]

# round decimals to 4 digits
score_cols = ["positive_prob", "neutral_prob", "negative_prob", "model_confidence", "overall_sentiment"]
df[score_cols] = df[score_cols].round(4)

display(df)

In [ ]:
df[['text', 'predicted_sentiment', 'model_confidence']]

In [ ]:
clean_df = df.copy()

print('Number of NaN values in reviews:', clean_df[clean_df['text'] == 'nan'].shape[0])
clean_df = clean_df[clean_df['text'] != 'nan']

print('Shape after removing empty reviews:', clean_df.shape)
clean_df.head()